1. 评估配置

In [1]:
grpo_model_path = "../output/grpo_mini_llama3_1"        # 经过冷启动和GRPO训练的模型
sft_model_path = grpo_model_path + "/cold_start_sft"    # 仅经过冷启动的模型

tokenizer_path = "../mini_tokenizer"
eval_data_path = "../data/grpo_data/eval.jsonl"
max_new_tokens = 512
batch_size = 8

2. 加载模型和数据

In [2]:
import gc
import json
import re

import torch
import mini_models  # 注册自定义模型
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# 加载评估数据
eval_data = []
with open(eval_data_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            eval_data.append(json.loads(line))
print(f"Loaded {len(eval_data)} eval samples")


# 加载模型
def load_model(model_path: str):
    model = AutoModelForCausalLM.from_pretrained(model_path).to(device)
    model.eval()
    print(f"Model loaded from: {model_path}")
    return model

Device: cuda
Loaded 300 eval samples


3. 解析与评估函数

In [3]:
def extract_json_from_response(text: str) -> str | None:
    """从 response 中提取唯一的 ```json ... ``` 包裹的 JSON 字符串"""
    matches = re.findall(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    if len(matches) == 1:
        return matches[0].strip()
    return None


def parse_prefilled_think_response(response_text: str) -> tuple[str | None, str | None]:
    """解析 prefill `<think>\n` 协议下的 continuation 输出。"""
    if response_text.count("<think>") != 0:
        return None, None
    if response_text.count("</think>") != 1:
        return None, None

    think_content, answer_text = response_text.split("</think>", maxsplit=1)
    return think_content.strip(), answer_text.lstrip()


def render_prefilled_response(response_text: str) -> str:
    """为了便于人工查看，将 prefilled `<think>` 补回展示文本。"""
    if "</think>" in response_text and "<think>" not in response_text:
        return "<think>\n" + response_text
    return response_text


def check_correct(response_text: str, ground_truth_response: str) -> dict:
    """
    检查模型输出是否正确:
    1. 是否正确闭合 prefilled `<think>`
    2. `</think>` 之后是否恰好存在一个 ```json...``` 包裹的输出
    3. 解析后的 JSON 与标准答案一致
    """
    result = {
        "has_think": False,
        "has_json_block": False,
        "json_parseable": False,
        "correct": False,
    }

    think_content, answer_text = parse_prefilled_think_response(response_text)
    if think_content is None:
        return result
    result["has_think"] = True

    json_str = extract_json_from_response(answer_text)
    if json_str is None:
        return result
    result["has_json_block"] = True

    try:
        parsed_output = json.loads(json_str)
    except (json.JSONDecodeError, ValueError):
        return result
    result["json_parseable"] = True

    gt_json_str = extract_json_from_response(ground_truth_response)
    if gt_json_str is None:
        return result
    try:
        gt_parsed = json.loads(gt_json_str)
    except (json.JSONDecodeError, ValueError):
        return result

    result["correct"] = parsed_output == gt_parsed
    return result


def generate_responses_batch(model, prompt_texts: list[str]) -> list[str]:
    """批量生成多条 prompt 的回复"""
    formatted_texts = []
    for prompt_text in prompt_texts:
        messages = [{"role": "user", "content": prompt_text}]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_think=True
        )
        formatted_texts.append(formatted)

    model_inputs = tokenizer(
        formatted_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)
    model_inputs.pop("token_type_ids", None)

    prompt_lengths = model_inputs["attention_mask"].sum(dim=1)
    context_window = int(getattr(model.config, "max_position_embeddings", max_new_tokens))
    remaining_context = int((context_window - prompt_lengths).min().item())
    effective_max_new_tokens = min(max_new_tokens, max(remaining_context, 0))
    if effective_max_new_tokens <= 0:
        return [""] * len(prompt_texts)

    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=effective_max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[:, model_inputs["input_ids"].shape[1]:]
    responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    return responses


def evaluate_model(model_path: str, model_name: str, batch_size: int = 8) -> dict:
    """按 batch 评估单个模型并返回统计结果"""
    model = load_model(model_path)
    results = []
    error_examples = []

    batch_starts = range(0, len(eval_data), batch_size)
    for start_idx in tqdm(batch_starts, desc=f"Evaluating {model_name}"):
        batch_samples = eval_data[start_idx:start_idx + batch_size]
        batch_prompts = [sample["prompt"] for sample in batch_samples]
        batch_responses = generate_responses_batch(model, batch_prompts)

        for sample, response_text in zip(batch_samples, batch_responses):
            check = check_correct(response_text, sample["response"])
            check["sample_id"] = sample.get("sample_id", "unknown")
            results.append(check)

            if not check["correct"]:
                error_examples.append({
                    "sample_id": sample.get("sample_id", "unknown"),
                    "prompt": sample["prompt"],
                    "model_output": response_text,
                    "ground_truth": sample["response"],
                    **check,
                })

    total = len(results)
    num_has_think = sum(r["has_think"] for r in results)
    num_has_json = sum(r["has_json_block"] for r in results)
    num_parseable = sum(r["json_parseable"] for r in results)
    num_correct = sum(r["correct"] for r in results)

    summary = {
        "model_name": model_name,
        "model_path": model_path,
        "total": total,
        "has_think": num_has_think,
        "has_json_block": num_has_json,
        "json_parseable": num_parseable,
        "correct": num_correct,
        "accuracy": num_correct / total if total else 0.0,
        "batch_size": batch_size,
    }

    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "summary": summary,
        "results": results,
        "error_examples": error_examples,
    }

4. 运行评估

In [4]:
print(f"Using batch_size={batch_size}")

grpo_eval = evaluate_model(grpo_model_path, "GRPO", batch_size=batch_size)
sft_eval = evaluate_model(sft_model_path, "SFT", batch_size=batch_size)

comparison_results = [
    grpo_eval["summary"],
    sft_eval["summary"],
]

print("Evaluation complete.")

Using batch_size=8
Model loaded from: ../output/grpo_mini_llama3_1


Evaluating GRPO: 100%|██████████| 38/38 [01:10<00:00,  1.86s/it]


Model loaded from: ../output/grpo_mini_llama3_1/cold_start_sft


Evaluating SFT: 100%|██████████| 38/38 [02:25<00:00,  3.83s/it]

Evaluation complete.


5. 准确率结果对比

In [5]:
for summary in comparison_results:
    total = summary["total"]
    print(f"\n{'=' * 60}")
    print(f"Model: {summary['model_name']}")
    print(f"Path: {summary['model_path']}")
    print(f"Total samples:          {total}")
    print(f"Closed prefilled <think>: {summary['has_think']} ({summary['has_think']/total*100:.1f}%)")
    print(f"Has ```json``` block:   {summary['has_json_block']} ({summary['has_json_block']/total*100:.1f}%)")
    print(f"JSON parseable:         {summary['json_parseable']} ({summary['json_parseable']/total*100:.1f}%)")
    print(f"Correct (accuracy):     {summary['correct']} ({summary['accuracy']*100:.1f}%)")

grpo_acc = grpo_eval["summary"]["accuracy"]
sft_acc = sft_eval["summary"]["accuracy"]

print(f"\n{'=' * 60}")
print(f"Accuracy gap (GRPO - SFT): {(grpo_acc - sft_acc) * 100:.2f}%")
print("Better model:", "GRPO" if grpo_acc > sft_acc else "SFT" if sft_acc > grpo_acc else "Tie")


Model: GRPO
Path: ../output/grpo_mini_llama3_1
Total samples:          300
Closed prefilled <think>: 300 (100.0%)
Has ```json``` block:   300 (100.0%)
JSON parseable:         297 (99.0%)
Correct (accuracy):     172 (57.3%)

Model: SFT
Path: ../output/grpo_mini_llama3_1/cold_start_sft
Total samples:          300
Closed prefilled <think>: 293 (97.7%)
Has ```json``` block:   291 (97.0%)
JSON parseable:         260 (86.7%)
Correct (accuracy):     86 (28.7%)

Accuracy gap (GRPO - SFT): 28.67%
Better model: GRPO


6. 错误样本示例

小模型本身的能力有限，此外，我们当前的奖励设计也较为简单，主要只从输出格式、思维链长度、可解析性和最终正确率等规则层面进行约束。在这种设置下，模型未必真正具备稳定的推理能力，`<think>` 标签中的内容也可能只是受格式和奖励驱动生成的中间文本，而不一定是真正意义上的思考过程，甚至可能与最终答案行为不一致。

In [6]:
for model_name, eval_result in [("GRPO", grpo_eval), ("SFT", sft_eval)]:
    print(f"\n{'#' * 80}")
    print(f"{model_name} error examples")

    for i, ex in enumerate(eval_result["error_examples"][:3], start=1):
        print(f"\n{'=' * 60}")
        print(f"Error Example {i} (sample_id={ex['sample_id']})")
        print(
            f"closed_prefilled_think={ex['has_think']}, has_json_block={ex['has_json_block']}, "
            f"json_parseable={ex['json_parseable']}"
        )
        print(f"\n【Prompt】: {ex['prompt'][:200]}")
        print(f"\n【Model Output】:\n{render_prefilled_response(ex['model_output'])[:500]}")
        print(f"\n【Ground Truth】:\n{ex['ground_truth']}")


################################################################################
GRPO error examples

Error Example 1 (sample_id=2)
closed_prefilled_think=True, has_json_block=True, json_parseable=True

【Prompt】: 请按要求修改这段JSON：删除所有值为null的字段，其余内容保持不变。JSON如下：{"title": "Project A", "description": null, "status": "planning", "deadline": null}

【Model Output】:
<think>
用户在原字段中字段的字段值是字段的字段值。原字段的字段值是字段的字段值。原字段的字段值是字段的字段值。原字段的字段值是字段的字段值。
</think>

```json
{"title": "Project A", "description": null, "status": "planning", "deadline": null, "deadline": null}

```

【Ground Truth】:
```json
{
  "title": "Project A",
  "status": "planning"
}
```

Error Example 2 (sample_id=4)
closed_prefilled_think=True, has_json_block=True, json_parseable=True

【Prompt】: 将下面JSON中的 enabled 改为布尔值true，其余不变：{"plugin": "cors", "version": "1.2.0", "enabled": "false"}

【Model Output】:
<think>
用户在原字段中字段的字段值是字段的字段值。原字段的字段值是字段的字段值。原字段的字段值是字段的字段值。原字段的字段值是字段的字段值。
</think>

```json
{"plugin": "cors", "version": "1.2.0", "enabled"